In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time,math

In [5]:
def scaled_dot_product_attention(q,k,v,mask=None):
  #q:[B,H,T,D_Q]
  #k:[B,H,t,D_K]
  #v:[B,H,T,D_V]
  d_k=q.size(-1)
  scores=q@k.transpose(-2,-1)/math.sqrt(d_k)
  if mask is not None:
    scores=scores.masked_fill(~mask,float('-inf'))
  att=F.softmax(scores,dim=-1)
  return scores@v,att

q=torch.randn(2,5,7,8)
k=torch.randn(2,5,7,8)
v=torch.randn(2,5,7,8)
scores,_=scaled_dot_product_attention(q,k,v)
print(scores)


tensor([[[[ 3.4194e+00,  1.9978e-02,  3.4033e-01,  8.8490e-01,  3.2423e-01,
            1.4691e-01,  2.6139e+00, -3.0739e-01],
          [ 9.6443e-01, -7.4733e-01,  2.1690e+00,  9.6580e-01, -3.9477e-01,
           -8.3088e-01, -1.5296e-02,  7.3921e-01],
          [ 3.0402e+00, -2.3226e+00,  1.9974e+00,  3.0080e+00,  1.9841e+00,
            3.5184e-01, -2.1690e+00,  1.7998e+00],
          [ 3.4446e+00,  1.1185e+00,  1.7367e+00, -1.6092e+00, -2.9774e-01,
           -8.9576e-01,  2.3813e+00, -9.9218e-01],
          [ 2.6127e+00, -5.3213e-01,  9.9259e-01,  7.6211e-01, -1.2900e+00,
           -1.9957e-01,  1.7842e+00,  1.1122e+00],
          [-2.8494e+00, -5.1915e-01,  6.3334e+00,  2.0887e+00, -5.6636e+00,
           -2.9030e+00,  1.4373e+00, -1.2653e-01],
          [ 1.0805e+00, -1.4360e+00, -2.4208e+00,  2.2777e+00,  4.7189e+00,
            1.5493e+00, -4.8822e+00,  1.9622e+00]],

         [[-6.5965e-01,  2.1302e+00, -3.7792e+00,  5.9917e+00, -5.4511e-01,
           -4.9328e+00,  8.0494e+

In [6]:
def my_softmax(x,dim=-1):
  max_val=torch.max(x,dim=dim,keepdim=True).values
  print("max_val:",max_val)
  exp_x=torch.exp(x-max_val)
  print(x-max_val)
  print("exp_x:",exp_x)
  sum_exp=torch.sum(exp_x,dim=dim,keepdim=True)
  print("sum_exp:",sum_exp)
  return exp_x/sum_exp
x=torch.tensor([[1,2],[3,4]])
out=my_softmax(x)
print(out)



max_val: tensor([[2],
        [4]])
tensor([[-1,  0],
        [-1,  0]])
exp_x: tensor([[0.3679, 1.0000],
        [0.3679, 1.0000]])
sum_exp: tensor([[1.3679],
        [1.3679]])
tensor([[0.2689, 0.7311],
        [0.2689, 0.7311]])


In [7]:
def my_softmax_2(x,dim=-1):
  exp_x=torch.exp(x)
  sum_exp=torch.sum(exp_x,dim=dim,keepdim=True)
  return exp_x/sum_exp
x=torch.tensor([[1,2],[3,4]])
print(my_softmax_2(x))

tensor([[0.2689, 0.7311],
        [0.2689, 0.7311]])


In [8]:
import torch

def softmax_naive(x, dim=-1):
    e = torch.exp(x)
    return e / e.sum(dim=dim, keepdim=True)

def softmax_stable(x, dim=-1):
    e = torch.exp(x - x.max(dim=dim, keepdim=True).values)
    return e / e.sum(dim=dim, keepdim=True)

# 1) 上溢：数值很大
x = torch.tensor([[1000., 1001.]])
print(torch.exp(x))          # tensor([[inf, inf]])
print(softmax_naive(x))      # tensor([[nan, nan]])  因为 inf/inf
print(softmax_stable(x))     # tensor([[0.2689, 0.7311]])  正确

# 2) 下溢：数值很负
x = torch.tensor([[-1000., -1001.]])
print(torch.exp(x))          # tensor([[0., 0.]])
print(softmax_naive(x))      # tensor([[nan, nan]])  因为 0/0
print(softmax_stable(x))     # tensor([[0.7311, 0.2689]])  正确

# 3) 临界点：float32 大约在 exp(88.7) 就溢出
print(torch.exp(torch.tensor(88.)))   # 1.6516e+38，还能表示
print(torch.exp(torch.tensor(89.)))   # inf

tensor([[inf, inf]])
tensor([[nan, nan]])
tensor([[0.2689, 0.7311]])
tensor([[0., 0.]])
tensor([[nan, nan]])
tensor([[0.7311, 0.2689]])
tensor(1.6516e+38)
tensor(inf)


In [9]:
#完整的放缩点击包含softmax的完整代码：
def my_softmax_3(x,dim=-1):
  max_val=torch.max(x,dim=dim,keepdim=True).values
  exp_x=torch.exp(x-max_val)
  return exp_x/torch.sum(exp_x,dim=dim,keepdim=True)
def scaled_dot_product_attention_1(q,k,v,mask=None):
  #q=[B,H,t,d_k],k=[B,H,t,d_k],v=[B,H,t,d_v]
  d_k=q.size(-1)
  scores=q@k.transpose(-2,-1)/math.sqrt(d_k)
  if mask is not None:
    scores=scores.masked_fill(~mask,float("-inf"))
  att=my_softmax_3(scores)
  return att@v,att
q=torch.randn(2,5,7,8)
k=torch.randn(2,5,7,8)
v=torch.randn(2,5,7,9)
T=7
mask=torch.tril(torch.ones(T,T,dtype=bool))
print("mask:",mask)
out,att=scaled_dot_product_attention_1(q,k,v)
print("out:",out)
print("att:",att)
mask_out,mask_att=scaled_dot_product_attention_1(q,k,v,mask)
print("causal_out:",mask_out)
print("causal_att",mask_att)

mask: tensor([[ True, False, False, False, False, False, False],
        [ True,  True, False, False, False, False, False],
        [ True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True]])
out: tensor([[[[ 8.1566e-01,  7.0833e-01,  6.1037e-01, -3.4902e-01,  2.8698e-01,
            5.4596e-01,  5.4372e-01, -9.8076e-01, -5.2335e-01],
          [-4.4970e-01, -3.1432e-02,  7.9500e-01, -7.4204e-01,  1.2413e+00,
           -9.7652e-01, -3.4631e-01, -4.3672e-01,  3.3583e-01],
          [ 1.1649e+00,  8.2049e-01,  7.0372e-01, -4.2006e-01,  4.3651e-01,
            7.0635e-01,  9.4517e-02, -1.0787e+00, -4.2168e-01],
          [ 5.5149e-01,  1.6425e-01,  3.5094e-01, -2.4493e-01,  3.7234e-01,
            7.4662e-02,  6.0212e-01, -7.4971e-01, -6.5928e-01],
          [-2.1

In [10]:
class MutleHeadAttention(nn.Module):
  def __init__(self,d_model,n_head,drop_out=0.01) -> None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.dk=d_model//n_head
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(drop_out)


  def split(self,x):
    B,T,_=x.shape
    return x.view(B,T,self.h,self.dk).transpose(1,2)

  def forward(self,x_q,x_kv,mask=None):
    q=self.split(self.W_q(x_q))
    k=self.split(self.W_k(x_kv))
    v=self.split(self.W_v(x_kv))
    out,self.lastatt=scaled_dot_product_attention_1(q,k,v,mask)
    out=self.dropout(out)
    B,H,T,dk=out.shape
    out=out.transpose(1,2).contiguous().view(B,T,H*dk)
    return self.W_o(out)


mha=MutleHeadAttention(32,4)
mha.eval() # Disable dropout for deterministic evaluation
x=torch.randn(2,6,32)
causal=torch.tril(torch.ones(6,6,dtype=bool))[None, None]
y1=mha(x,x,causal)
x2=x.clone();x2[:,4:]=torch.randn(2,2,32)
y2=mha(x2,x2,causal)
print(y1.shape)
assert torch.allclose(y1[:, :4], y2[:, :4], atol=1e-6)
print("causality OK")

torch.Size([2, 6, 32])
causality OK


In [11]:
class multiheadattention(nn.Module):
  def __init__(self,d_model,n_head,drop_out=0.01) -> None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.dk=d_model//n_head
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)
    self.dropout=nn.Dropout(drop_out)

  def split(self,x):
    B,T,_=x.shape
    x=x.view(B,T,self.h,self.dk).transpose(1,2)
    return x

  def forward(self,q,k,v,mask=None):
    q=self.split(self.W_q(q))
    k=self.split(self.W_k(k))
    v=self.split(self.W_v(v))
    out,self.lastatt=scaled_dot_product_attention_1(q,k,v,mask)
    out=self.dropout(out)
    B,H,T,dk=out.shape
    out=out.transpose(1,2).contiguous().view(B,T,H*dk)
    return self.W_o(out)

mha1=multiheadattention(32,4).eval()
x=torch.randn(2,6,32)
causal=torch.tril(torch.ones(6,6,dtype=bool))
y1=mha1(x,x,x,causal)
print(y1.shape)
x2=x.clone();x2[:,4:]=torch.randn(2,2,32)
y2=mha1(x2,x2,x2,causal)
assert torch.allclose(y1[:,:4],y2[:,:4],atol=1e-6)
print("causality OK")

torch.Size([2, 6, 32])
causality OK


In [12]:
#位置编码
class SinusoidalPE(nn.Module):
  def __init__(self,d_model,max_len=2048) -> None:
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    pos=torch.arange(max_len).unsqueeze(1).float()
    div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(pos*div)
    pe[:,1::2]=torch.cos(pos*div)
    self.register_buffer("pe",pe)
  def forward(self,x):
    x=x+self.pe[:x.size(1)]
    return x

pe=SinusoidalPE(64)
print(pe.pe.shape)
print(pe.pe)
print(pe)


torch.Size([2048, 64])
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  6.8156e-01,  ...,  1.0000e+00,
          1.3335e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.9748e-01,  ...,  1.0000e+00,
          2.6670e-04,  1.0000e+00],
        ...,
        [ 1.7590e-01, -9.8441e-01,  4.2275e-01,  ...,  9.3460e-01,
          2.6934e-01,  9.6305e-01],
        [-7.3331e-01, -6.7989e-01,  9.2701e-01,  ...,  9.3454e-01,
          2.6947e-01,  9.6301e-01],
        [-9.6832e-01,  2.4972e-01,  9.3397e-01,  ...,  9.3448e-01,
          2.6959e-01,  9.6297e-01]])
SinusoidalPE()


In [13]:
class sinusoidialPE1(nn.Module):
  def __init__(self,d_model,max_len=4096) -> None:
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    pos=torch.arange(max_len).unsqueeze(1).float()
    div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(pos*div)
    pe[:,1::2]=torch.cos(pos*div)
    self.register_buffer("pe",pe)

  def forward(self,x):
    return x+pe[:x.size(1)]
PE1=sinusoidialPE1(64)
print(PE1.pe.shape)




torch.Size([4096, 64])


In [14]:
#FFN
class FeedForward(nn.Module):
  def __init__(self,d_model,d_dff,dropout=0.01) -> None:
    super().__init__()
    self.net=nn.Sequential(nn.Linear(d_model,d_dff),nn.GELU(),nn.Linear(d_dff,d_model),nn.Dropout(dropout))
  def forward(self,x):return self.net(x)

In [15]:
#EncoderLayer
class EncoderLayer(nn.Module):
  def __init__(self,d_model,n_head,d_dff,drop_out=0.01) -> None:
    super().__init__()
    self.att=MutleHeadAttention(d_model,n_head,drop_out)
    self.ffn=FeedForward(d_model,d_dff,drop_out)
    self.lay1=nn.LayerNorm(d_model)
    self.lay2=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(drop_out)


  def forward(self,x,src_mask):
    x=self.lay1(x)
    x=x+self.dropout(self.att(x,x,src_mask))
    x=self.lay2(x)
    x=x+self.dropout(self.ffn(x))
    return x

In [16]:
#DecoderLayer
class DecoderLayer(nn.Module):
  def __init__(self,d_model,n_head,d_dff,drop_out=0.01) -> None:
    super().__init__()
    self.self_attn=MutleHeadAttention(d_model,n_head,drop_out)
    self.cross_attn=MutleHeadAttention(d_model,n_head,drop_out)
    self.ffn=FeedForward(d_model,d_dff,drop_out)
    self.ly1=nn.LayerNorm(d_model)
    self.ly2=nn.LayerNorm(d_model)
    self.ly3=nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(drop_out)
  def forward(self,x,memory,tgt_mask,src_mask):
    x=x+self.dropout(self.self_attn(self.ly1(x),self.ly1(x),tgt_mask))
    x=self.ly2(x)
    x=x+self.dropout(self.cross_attn(x,memory,src_mask))
    x=x+self.dropout(self.ffn(self.ly3(x)))
    return x


In [17]:
#Transformer
class Transformer(nn.Module):
  def __init__(self,src_voc,tgt_voc,d_model=128,n_head=4,drop_out=0.01,pad_id=0,max_len=2048,n_layer=3,d_dff=512) -> None:
    super().__init__()
    self.d_model,self.padId=d_model,pad_id
    self.PE=SinusoidalPE(d_model,max_len)
    self.ln1=nn.LayerNorm(d_model)
    self.ln2=nn.LayerNorm(d_model)
    self.src_emb=nn.Embedding(src_voc,d_model,padding_idx=pad_id)
    self.tgt_emb=nn.Embedding(tgt_voc,d_model,padding_idx=pad_id)
    self.dropout=nn.Dropout(drop_out)
    self.enc=nn.ModuleList(EncoderLayer(d_model,n_head,d_dff,drop_out) for _ in range(n_layer))
    self.dec=nn.ModuleList(DecoderLayer(d_model,n_head,d_dff,drop_out) for _ in range(n_layer))
    self.out=nn.Linear(d_model,tgt_voc)

  def make_mask(self,src_voc,tgt_voc):
    src_mask=(src_voc!=self.padId)[:,None,None,:]
    Tt=tgt_voc.size(1)
    causal=torch.tril(torch.ones(Tt,Tt,dtype=bool,device=tgt_voc.device))
    tgt_mask=(tgt_voc!=self.padId)[:,None,None,:]&causal
    return src_mask,tgt_mask

  def encode(self,src,src_mask):
    x=self.dropout(self.PE(self.src_emb(src)*math.sqrt(self.d_model)))
    for layer in self.enc:x=layer(x,src_mask)
    return self.ln1(x)

  def decode(self,tgt,memory,src_mask,tgt_mask):
    x=self.dropout(self.PE(self.tgt_emb(tgt)*math.sqrt(self.d_model)))
    for layer in self.dec:x=layer(x,memory,tgt_mask,src_mask)
    return self.out(self.ln2(x))

  def forward(self,src,tgt):
    src_mask,tgt_mask=self.make_mask(src,tgt)
    out=self.decode(tgt,self.encode(src,src_mask),src_mask,tgt_mask)
    return out

model=Transformer(20,20)
src=torch.randint(3,20,(2,8))
tgt=torch.randint(3,20,(2,9))
print(model(src, tgt).shape, "params:", sum(p.numel() for p in model.parameters()))

torch.Size([2, 9, 20]) params: 1396756


In [18]:
import urllib.request
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = urllib.request.urlopen(url, timeout=10).read().decode("utf-8")
except Exception as e:
    print("download failed, use fallback:", e)
    text = "hello transformer, attention is all you need. " * 200

chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print("chars:", len(text), "vocab:", len(chars))

BLOCK = 128
def get_batch(split, B=64):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - BLOCK - 1, (B,))
    x = torch.stack([d[i:i + BLOCK] for i in ix])
    y = torch.stack([d[i + 1:i + BLOCK + 1] for i in ix])      # 右移一位
    return x.to(device), y.to(device)

chars: 1115394 vocab: 65


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GPTBlocks(nn.Module):
  def __init__(self,d,h,dropout=0.01)->None:
    super().__init__()
    self.ln1,self.ln2=nn.LayerNorm(d),nn.LayerNorm(d)
    self.attn=MutleHeadAttention(d,h,dropout)
    self.ffn=FeedForward(d,4*d,dropout)

  def forward(self,x,mask):
    a=self.ln1(x)
    x=x+self.attn(a,a,mask)
    return x+self.ffn(self.ln2(x))

class MiniGPT(nn.Module):
  def __init__(self,vocab,d=192,h=6,block=BLOCK,n_layers=4,dropout=0.01)->None:
    super().__init__()
    self.block=block
    self.tok=nn.Embedding(vocab,d)
    self.pos=nn.Embedding(block,d)
    self.blocks=nn.ModuleList(GPTBlocks(d,h,dropout) for _ in range(n_layers))
    self.ln=nn.LayerNorm(d)
    self.head=nn.Linear(d,vocab, bias=False)
    self.head.weight=self.tok.weight
    self.register_buffer("mask",torch.tril(torch.ones(block,block,dtype=bool))[None,None])

  def forward(self,idx,target=None):
    B,T=idx.shape
    x=self.tok(idx)+self.pos(torch.arange(T,device=idx.device))
    for b in self.blocks:x=b(x,self.mask[:,:,:T,:T])
    logits=self.head(self.ln(x))
    loss=None if target is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)),target.reshape(-1))
    return logits,loss

  @torch.no_grad()
  def generate(self,idx,max_new=200,temperature=1.0,top_k=None):
    self.eval()
    for _ in range(max_new):
      logits,_=self(idx[:,-self.block:])
      logits=logits[:,-1]/temperature
      if top_k is not None:
        v,_=torch.topk(logits,top_k)
        logits[logits<v[:,[-1]]]=float("-inf")
      nxt=torch.multinomial(F.softmax(logits,-1),1)
      idx=torch.cat([idx,nxt],1)
    return idx

gpt=MiniGPT(len(chars)).to(device)
opt=torch.optim.AdamW(gpt.parameters(),lr=3.0e-4,weight_decay=0.1)
steps=1000

t0=time.time()
for step in range(1,steps+1):
  gpt.train()
  x,y=get_batch("train")
  _,loss=gpt(x,y)
  opt.zero_grad();loss.backward()
  torch.nn.utils.clip_grad_norm_(gpt.parameters(),1.0)
  opt.step()
  if step % 250 == 0:
        gpt.eval()
        with torch.no_grad(): _, vl = gpt(*get_batch("val"))
        print(f"step {step} train {loss.item():.3f} val {vl.item():.3f}  {time.time()-t0:.0f}s")
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print("".join(itos[i] for i in gpt.generate(start, 300, temperature=0.8, top_k=40)[0].tolist()))

step 250 train 2.614 val 2.572  16s
step 500 train 2.450 val 2.445  31s
step 750 train 2.335 val 2.307  46s
step 1000 train 2.166 val 2.194  62s

ORIS:
The Hut son ste cemed itherk athe grith thall:
And, not too mo the at grent con hist,
And stay on arnitende sue freat; tred wour onch:
eake in he her no bed, ch ce en.
She And con toustiriknard ind mone fres in'd cinfld.

Thercond Ed:
Whild will of thist they:

Ser of beill by wor If hour st i
